# 09 Area Condo Content Feature Econometrics: V2 Interpretation Patch

This notebook patches the robustness reporting of notebook 09 without changing the estimand or refitting the original model ladder.

- Outcome: `cited`
- Unit: one surfaced source appearance
- Estimand: citation probability conditional on a source already being surfaced
- Interpretation: observational association, not causal and not web-wide

The patch does not use answer-derived variables, does not introduce final enriched page type as a main control, and does not overwrite raw inputs or original model outputs.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

CODE_ROOT = Path.cwd().resolve()
if not (CODE_ROOT / "src").exists() and (CODE_ROOT.parent / "src").exists():
    CODE_ROOT = CODE_ROOT.parent
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from src.econometrics_eda_v2.content_feature_interpretation_patch import (
    run_content_feature_interpretation_patch,
)
from src.econometrics_eda_v2.paths import topic_output_dir

PACKAGE = topic_output_dir() / "content_econometrics_ai_package"
TABLES = PACKAGE / "tables/09_content_feature_econometrics/interp_patch"
FIGURES = PACKAGE / "figures/09_content_feature_econometrics/interp_patch"
REPORTS = PACKAGE / "reports/09_content_feature_econometrics/interp_patch"

result = run_content_feature_interpretation_patch(PACKAGE)
display(pd.DataFrame([result]).T.rename(columns={0: "value"}))

def table(name, n=None):
    frame = pd.read_csv(TABLES / name, low_memory=False)
    return frame if n is None else frame.head(n)

def show_plot(name):
    display(pio.read_json(FIGURES / name.replace(".html", ".plotly.json")))

## 1. Focal-term standard-error comparison

In [ ]:
display(table("focal_term_se_comparison.csv"))
display(table("focal_term_se_stability_summary.csv"))
show_plot("focal_term_se_comparison_forest.html")

The same coefficient is compared under HC3, prompt-cluster, URL-cluster, and two-way prompt-by-URL clustered inference. A result is not treated as definitive when interval conclusions depend on one covariance estimator.

## 2. Two-way cluster warning audit

In [ ]:
display(table('two_way_cluster_warning_audit.csv'))

Two-way clustered covariance can be unstable in high-dimensional fixed-effect models with repeated prompts and URLs. Negative diagonal variances indicate that some reported SEs are not valid for affected terms. Therefore, focal content estimates should be checked against HC3, prompt-cluster, and URL-cluster alternatives.

## 3. Robustness classification

In [ ]:
display(table('focal_feature_robustness_classification.csv'))

The classification is intentionally conservative:

- `suggestive`: direction is informative, but uncertainty or attenuation prevents a definitive statement.
- `unstable`: important specifications change sign, magnitude, or interval conclusion.
- `descriptive_only`: the feature is a diagnostic/control rather than substantive writing quality.

## 4. Domain-FE attenuation

In [ ]:
display(table("domain_fe_attenuation_summary.csv"))
show_plot("domain_fe_attenuation_focal_terms.html")

Heading-count categories show large negative associations in prompt-FE models, but these estimates attenuate strongly under domain fixed effects. This suggests domain/template or page-function differences may explain much of the pattern.

## 5. Outlier sensitivity

In [ ]:
display(table('outlier_sensitivity_focal_terms.csv'))

Page length is sensitive to extreme word-count tails. The preferred estimate is small and imprecise, while removing the top 1% word-count tail changes its magnitude and interval conclusion.

## 6. Revised interpretation report

In [ ]:
display(Markdown(
    (REPORTS / "09_content_feature_econometrics_report_v2_interpretation_patch.md").read_text()
))

## 7. Patched minimum reporting table

In [ ]:
display(table('09_minimum_reporting_table_v2_interpretation_patch.csv'))

The final interpretation buckets are `suggestive_positive`, `domain_template_confounded`, `unstable_diagnostic`, `extraction_quality_control`, and `insufficient_support`.

## 8. Executive summary

In [ ]:
display(Markdown(
    (REPORTS / "09_content_feature_econometrics_executive_summary_v2.md").read_text()
))

## 9. Next feature layer

In [ ]:
display(table('next_feature_layer_priority_plan.csv'))

All proposed next-layer features use page text, page structure, links, structured data, or prompt text only. Answer text and citation outcomes must not define the features.

## 10. Final status

In [ ]:
print(f"number of focal terms checked: {result['number_of_focal_terms_checked']}")
print(f"number of models included in SE comparison: {result['number_of_models_included_in_se_comparison']}")
print(f"number of two-way cluster warnings: {result['number_of_two_way_cluster_warnings']}")
print(f"number of features classified as suggestive: {result['number_of_features_classified_as_suggestive']}")
print(
    "number of features classified as domain/template-confounded: "
    f"{result['number_of_features_classified_as_domain_template_confounded']}"
)
print(
    "number of features classified as unstable/diagnostic: "
    f"{result['number_of_features_classified_as_unstable_diagnostic']}"
)
print(f"path to revised report: {result['revised_report']}")
print(f"path to revised minimum reporting table: {result['revised_minimum_reporting_table']}")
print(f"final status: {result['final_status']}")